# Text Classification with ClassifierDL - part4

**Relevant blogpost:** https://towardsdatascience.com/text-classification-in-spark-nlp-with-bert-and-universal-sentence-encoders-e644d618ca32

# SentimentDL Classifier

In [1]:
import sparknlp

spark = sparknlp.start(gpu = False) #True) # for GPU training >> sparknlp.start(gpu = True) # for Spark 2.3 =>> sparknlp.start(spark23 = True)

from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
import pandas as pd


print("Spark NLP version", sparknlp.version())
print("Apache Spark version:", spark.version)

spark

Spark NLP version 3.3.2
Apache Spark version: 3.1.2


see also here >> `https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/jupyter/training/english/classification/SentimentDL_train_multiclass_sentiment_classifier.ipynb`

In [ ]:
#!wget -O aclimdb_train.csv https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/sentiment-corpus/aclimdb/aclimdb_train.csv

In [13]:

trainDataset = spark.read \
      .option("header", True) \
      .csv("data/aclimdb_train.csv")

trainDataset.show()


+--------------------+--------+
|                text|   label|
+--------------------+--------+
|This is an Excell...|positive|
|The Sarah Silverm...|positive|
|"Prom Night" is a...|negative|
|So often a band w...|positive|
|"Pet Sematary" is...|positive|
|I watched the fil...|negative|
|Boy this movie ha...|negative|
|Checking the spoi...|negative|
|Despite its rathe...|positive|
|Absolute masterpi...|positive|
|The tweedy profes...|positive|
|A movie best summ...|negative|
|Take young, prett...|negative|
|For months I've b...|negative|
|"Batman: The Myst...|positive|
|Well, it was funn...|negative|
|I have seen the s...|positive|
|Brainless film ab...|negative|
|Leave it to geniu...|negative|
|Seven Pounds star...|positive|
+--------------------+--------+
only showing top 20 rows



In [ ]:
#!wget -O aclimdb_test.csv https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/resources/en/sentiment-corpus/aclimdb/aclimdb_test.csv


In [18]:
testDataset = spark.read \
      .option("header", True) \
      .csv("data/aclimdb_test.csv")

testDataset.show()

+--------------------+--------+
|                text|   label|
+--------------------+--------+
|The Second Woman ...|negative|
|In my opinion the...|positive|
|I am listening to...|positive|
|Before I speak my...|negative|
|ManBearPig is a p...|positive|
|A buddy and I wen...|negative|
|It is incredible ...|negative|
|Dire! Dismal! Awf...|negative|
|HLOTS was an outs...|positive|
|This is just one ...|negative|
|This movie had th...|negative|
|The 80s were over...|positive|
|The tunes are the...|positive|
|Having recently s...|positive|
|My favorite film ...|positive|
|This movie just m...|positive|
|This is the worst...|negative|
|This was a nice f...|positive|
|I don't know, may...|negative|
|After wasting 2 h...|negative|
+--------------------+--------+
only showing top 20 rows



In [21]:
# actual content is inside description column
document = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

use = UniversalSentenceEncoder.pretrained("tfhub_use_lg", "en") \
    .setInputCols("document") \
    .setOutputCol("sentence_embeddings")

# the classes/labels/categories are in category column
sentimentdl = SentimentDLApproach()\
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("class")\
    .setLabelColumn("label")\
    .setMaxEpochs(5)\
    .setEnableOutputLogs(True)

pipeline = Pipeline(
    stages = [
        document,
        use,
        sentimentdl
    ])


tfhub_use_lg download started this may take some time.
Approximate size to download 753.3 MB
[OK!]


In [22]:
#%%time
pipelineModel = pipeline.fit(trainDataset)

In [23]:
result = pipelineModel.transform(testDataset)

In [26]:
result_df = result.select('text','label',"class.result").toPandas()

In [27]:
result_df.head(10)

,text,label,result
0,The Second Woman is about the story of a myste...,negative,[positive]
1,"In my opinion the directing, editing, lighting...",positive,[positive]
2,"I am listening to Istanbul, intent, my eyes cl...",positive,[positive]
3,"Before I speak my piece, I would like to make ...",negative,[neutral]
4,ManBearPig is a pretty funny episode of South ...,positive,[positive]
5,A buddy and I went to see this movie when it c...,negative,[negative]
6,It is incredible that there were two films wit...,negative,[negative]
7,Dire! Dismal! Awful! Laughable! Disappointing!...,negative,[negative]
8,"HLOTS was an outstanding series, its what NYPD...",positive,[positive]
9,This is just one of those films which cannot j...,negative,[negative]
